# 04 — Evaluation and Comparison

Load the trained scGPT checkpoint, run it on the test set, and compare it against the baselines from notebook 02.

**This notebook can be run independently** after training completes — no need to re-train.

**What we'll compute:**
- Per-perturbation Pearson correlation (all genes)
- Pearson on DE genes only (the signal metric)
- Pearson on delta from control (removes baseline expression effect)
- Side-by-side comparison table: scGPT vs DeltaMean vs MeanControl

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from gears import PertData

from src.config import Config
from src.model_utils import load_vocab_and_gene_ids, build_model
from src.train import eval_perturb
from src.baselines import MeanControlBaseline, DeltaMeanBaseline
from src.evaluate import compute_metrics, format_baseline_results, filter_ctrl_from_results

## 1. Load data and checkpoint

In [ ]:
config = Config(
    data_dir='../data',
    pretrained_dir='../pretrained/scGPT_human',
    use_fast_transformer=False,
    amp=False,
)
device = config.device

pert_data = PertData(config.data_dir)
pert_data.load(data_name='adamson')
pert_data.prepare_split(split='simulation', seed=1)
pert_data.get_dataloader(batch_size=64, test_batch_size=64)

genes = pert_data.adata.var['gene_name'].tolist()
n_genes = len(genes)
vocab, gene_ids = load_vocab_and_gene_ids(
    config.pretrained_dir, genes, config.special_tokens
)

# Load the trained checkpoint
run_path_file = Path('../outputs/latest_run.txt')
if run_path_file.exists():
    save_dir = Path(run_path_file.read_text().strip())
else:
    # Manually set the path if needed:
    # save_dir = Path('../outputs/runs/adamson-Nov10-14-30')
    raise FileNotFoundError(
        'Run notebook 03 first, or set save_dir manually'
    )

print(f'Loading checkpoint from: {save_dir}')
model = build_model(config, vocab)
checkpoint = torch.load(save_dir / 'best_model.pt', map_location='cpu')
model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()
print('Model loaded.')

## 2. Run scGPT on test set

In [ ]:
adata = pert_data.adata
ctrl_adata = adata[adata.obs['condition'] == 'ctrl']
ctrl_mean = np.asarray(ctrl_adata.X.mean(axis=0)).flatten()

train_conditions = pert_data.set2conditions['train']
train_adata = adata[adata.obs['condition'].isin(train_conditions)]
test_perts = pert_data.set2conditions['test']

ctrl_baseline = MeanControlBaseline()
ctrl_baseline.fit(ctrl_adata)
ctrl_preds = ctrl_baseline.predict(test_perts)
ctrl_results = format_baseline_results(ctrl_preds, pert_data)

delta_baseline = DeltaMeanBaseline()
delta_baseline.fit(train_adata, ctrl_adata)
delta_preds = delta_baseline.predict(test_perts)
delta_results = format_baseline_results(delta_preds, pert_data)

print('Baselines recomputed.')

all_metrics = {}
for name, results in [
    ('scGPT', scgpt_results),
    ('DeltaMean', delta_results),
    ('MeanControl', ctrl_results),
]:
    clean = filter_ctrl_from_results(results)
    all_metrics[name] = compute_metrics(clean, ctrl_adata)

comparison_df = pd.DataFrame(all_metrics).T[
    ['pearson', 'pearson_delta', 'pearson_de', 'pearson_de_delta']
]
print('=== Final Comparison Table ===')
print(comparison_df.to_string(float_format='{:.4f}'.format))
print()
print('Key metric: pearson_de_delta')
print('Expected (from scGPT paper, Adamson):')
print('  scGPT: ~0.80')
print('  DeltaMean: lower (this model has no gene-specific knowledge)')

In [ ]:
adata = pert_data.adata
ctrl_adata = adata[adata.obs['condition'] == 'ctrl']
ctrl_mean = np.asarray(ctrl_adata.X.mean(axis=0)).flatten()

train_conditions = pert_data.split['train'] + ['ctrl']
train_adata = adata[adata.obs['condition'].isin(train_conditions)]
test_perts = pert_data.split['test']

ctrl_baseline = MeanControlBaseline()
ctrl_baseline.fit(ctrl_adata)
ctrl_preds = ctrl_baseline.predict(test_perts)
ctrl_results = format_baseline_results(ctrl_preds, pert_data)

delta_baseline = DeltaMeanBaseline()
delta_baseline.fit(train_adata, ctrl_adata)
delta_preds = delta_baseline.predict(test_perts)
delta_results = format_baseline_results(delta_preds, pert_data)

print('Baselines recomputed.')

## 4. Comparison table

**Metric guide:**
- `pearson` — overall correlation across all genes (misleadingly high for all models)
- `pearson_delta` — correlation of *changes* from control (harder, more meaningful)
- `pearson_de` — correlation on DE genes only (the standard signal metric)

In [ ]:
scgpt_de_delta = all_metrics['scGPT']['pearson_de_delta']
baseline_de_delta = all_metrics['DeltaMean']['pearson_de_delta']
ctrl_de_delta = all_metrics['MeanControl']['pearson_de_delta']

assert scgpt_de_delta > baseline_de_delta, \
    f'scGPT ({scgpt_de_delta:.3f}) should beat DeltaMean ({baseline_de_delta:.3f}) on pearson_de_delta'
assert scgpt_de_delta > ctrl_de_delta, 'scGPT should beat MeanControl'

print('Phase 5 validation PASSED!')
print(f'  scGPT pearson_de_delta = {scgpt_de_delta:.4f}')
print(f'  DeltaMean pearson_de_delta = {baseline_de_delta:.4f}')
print(f'  MeanControl pearson_de_delta = {ctrl_de_delta:.4f}')
print(f'  scGPT improvement over DeltaMean: {scgpt_de_delta - baseline_de_delta:+.4f}')

# Save final results
comparison_df.to_csv('../outputs/final_comparison.csv')
print('\nFinal comparison saved to outputs/final_comparison.csv')

## 5. Per-perturbation breakdown

In [ ]:
scgpt_per_pert = compute_per_perturbation_metrics(scgpt_results)
delta_per_pert = compute_per_perturbation_metrics(delta_results)

# Merge for comparison
merged = scgpt_per_pert.merge(
    delta_per_pert, on='perturbation', suffixes=('_scGPT', '_DeltaMean')
)
merged['scgpt_improvement'] = merged['pearson_de_scGPT'] - merged['pearson_de_DeltaMean']
merged = merged.sort_values('scgpt_improvement', ascending=False)

print('Perturbations where scGPT most outperforms DeltaMean:')
print(merged[['perturbation', 'pearson_de_scGPT', 'pearson_de_DeltaMean', 'scgpt_improvement']]
      .head(5).to_string(index=False, float_format='{:.3f}'.format))

print('\nPerturbations where DeltaMean is better (scGPT underperforms):')
print(merged[['perturbation', 'pearson_de_scGPT', 'pearson_de_DeltaMean', 'scgpt_improvement']]
      .tail(5).to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Scatter: per-perturbation scGPT vs DeltaMean pearson_de
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    merged['pearson_de_DeltaMean'],
    merged['pearson_de_scGPT'],
    alpha=0.7, s=50
)
lims = [min(merged[['pearson_de_scGPT', 'pearson_de_DeltaMean']].min()), 1.0]
ax.plot(lims, lims, 'r--', label='equal performance')
ax.set_xlabel('DeltaMean pearson_de')
ax.set_ylabel('scGPT pearson_de')
ax.set_title('Per-perturbation DE Pearson: scGPT vs DeltaMean baseline\n'
             '(points above diagonal = scGPT wins)')
ax.legend()
plt.tight_layout()
plt.show()

n_wins = (merged['pearson_de_scGPT'] > merged['pearson_de_DeltaMean']).sum()
print(f'scGPT wins on {n_wins}/{len(merged)} perturbations')

## 6. Visualize predictions for specific perturbations

In [ ]:
# Pick the perturbation where scGPT improves most over baseline
best_pert = merged.iloc[0]['perturbation']
mask = scgpt_results['pert_cat'] == best_pert

scgpt_pred = scgpt_results['pred'][mask].mean(axis=0)
true_expr = scgpt_results['truth'][mask].mean(axis=0)
delta_pred = delta_preds[best_pert]

delta_from_ctrl = true_expr - ctrl_mean
top_de = np.argsort(np.abs(delta_from_ctrl))[-50:]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, pred) in zip(axes, [('scGPT', scgpt_pred), ('DeltaMean', delta_pred)]):
    ax.scatter(true_expr[top_de], pred[top_de], alpha=0.7, s=30)
    lo, hi = true_expr[top_de].min(), true_expr[top_de].max()
    ax.plot([lo, hi], [lo, hi], 'r--', alpha=0.5)
    r = np.corrcoef(true_expr[top_de], pred[top_de])[0, 1]
    ax.set_title(f'{name}\nPearson (top 50 DE genes) = {r:.3f}\n{best_pert}')
    ax.set_xlabel('True expression')
    ax.set_ylabel('Predicted expression')

plt.tight_layout()
plt.show()

## 7. Phase 5 Validation Checks

In [ ]:
scgpt_de = all_metrics['scGPT']['pearson_de']
baseline_de = all_metrics['DeltaMean']['pearson_de']
ctrl_de = all_metrics['MeanControl']['pearson_de']

assert scgpt_de > baseline_de, \
    f'scGPT ({scgpt_de:.3f}) should beat DeltaMean ({baseline_de:.3f}) on pearson_de'
assert scgpt_de > ctrl_de, 'scGPT should beat MeanControl'
assert comparison_df is not None

print('Phase 5 validation PASSED!')
print(f'  scGPT pearson_de = {scgpt_de:.4f}')
print(f'  DeltaMean pearson_de = {baseline_de:.4f}')
print(f'  Improvement: {scgpt_de - baseline_de:+.4f}')

# Save final results
comparison_df.to_csv('../outputs/final_comparison.csv')
print('\nFinal comparison saved to outputs/final_comparison.csv')